<a href="https://colab.research.google.com/github/Baze-Bai/XAI/blob/Adaversarial-Patch/AP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AIPI 590 - XAI | Adversarial Patch
### Zejun(Baze) Bai

# Reference
Dataset: https://raw.githubusercontent.com/phlippe/saved_models/main/tutorial10/

Model: Resnet34

# Import required packages and Set up

In [57]:
## Standard libraries
import os
import json
import math
import time
import numpy as np
import scipy.linalg
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os

## Imports for plotting
import matplotlib.pyplot as plt
%matplotlib inline
from IPython.display import set_matplotlib_formats
set_matplotlib_formats('svg', 'pdf') # For export
from matplotlib.colors import to_rgb
import matplotlib
matplotlib.rcParams['lines.linewidth'] = 2.0
import seaborn as sns
sns.set()

## Progress bar
from tqdm.notebook import tqdm

## PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
import torch.optim as optim
# Torchvision
import torchvision
from torchvision.datasets import CIFAR10
from torchvision import transforms
# PyTorch Lightning
try:
    import pytorch_lightning as pl
except ModuleNotFoundError: # Google Colab does not have PyTorch Lightning installed by default. Hence, we do it here if necessary
    !pip install --quiet pytorch-lightning>=1.4
    import pytorch_lightning as pl
from pytorch_lightning.callbacks import LearningRateMonitor, ModelCheckpoint

# Path to the folder where the datasets are/should be downloaded (e.g. MNIST)
DATASET_PATH = "./data"
# Path to the folder where the pretrained models are saved
CHECKPOINT_PATH = "./data"

# Setting the seed
pl.seed_everything(42)

# Ensure that all operations are deterministic on GPU (if used) for reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Fetching the device that will be used throughout this notebook
device = torch.device("cpu") if not torch.cuda.is_available() else torch.device("cuda:0")

<ipython-input-57-386cbea7a6b8>:17: DeprecationWarning: `set_matplotlib_formats` is deprecated since IPython 7.23, directly use `matplotlib_inline.backend_inline.set_matplotlib_formats()`
  set_matplotlib_formats('svg', 'pdf') # For export
INFO:lightning_fabric.utilities.seed:Seed set to 42


# Download the Dataset for generating patch

In [13]:
import urllib.request
from urllib.error import HTTPError
import zipfile
# Github URL where the dataset is stored for this tutorial
base_url = "https://raw.githubusercontent.com/phlippe/saved_models/main/tutorial10/"
# Files to download
pretrained_files = [(DATASET_PATH, "TinyImageNet.zip"), (CHECKPOINT_PATH, "patches.zip")]
# Create checkpoint path if it doesn't exist yet
os.makedirs(DATASET_PATH, exist_ok=True)
os.makedirs(CHECKPOINT_PATH, exist_ok=True)

# For each file, check whether it already exists. If not, try downloading it.
for dir_name, file_name in pretrained_files:
    file_path = os.path.join(dir_name, file_name)
    if not os.path.isfile(file_path):
        file_url = base_url + file_name
        print(f"Downloading {file_url}...")
        try:
            urllib.request.urlretrieve(file_url, file_path)
        except HTTPError as e:
            print("Something went wrong. Please try to download the file from the GDrive folder, or contact the author with the full output including the following error:\n", e)
        if file_name.endswith(".zip"):
            print("Unzipping file...")
            with zipfile.ZipFile(file_path, 'r') as zip_ref:
                zip_ref.extractall(file_path.rsplit("/",1)[0])

Unzipping file...
Unzipping file...


# Set up the parameters

In [15]:
# Configuration parameters
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
snake_class = 56  # Class index for "boa constrictor" in ImageNet
patch_size = 100  # Patch size
epochs = 500      # Increase training epochs to handle small dataset
lr = 0.01
batch_size = 6    # Matching the dataset size

# Download the pretrained model

In [16]:

model = models.resnet34(pretrained=True).eval().to(device)
for param in model.parameters():
    param.requires_grad = False

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


# Preprocessing images

In [20]:
# Define the data class
class SnakeDataset(Dataset):
    def __init__(self, root_dir):
        self.root_dir = root_dir
        self.image_files = [f for f in os.listdir(root_dir) if f.endswith(('.jpg', '.png,', '.JPEG'))]
        self.transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.image_files[idx])
        image = Image.open(img_path).convert('RGB')
        return self.transform(image)


In [21]:
# upload the dataset
dataset = SnakeDataset(root_dir='/content/data/TinyImageNet/n01735189')
# preprocess the images
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Initialize the patch
patch = torch.rand((3, patch_size, patch_size), requires_grad=True, device=device)

# Define the application function

In [31]:
# Improved patch application function
def apply_patch(images):
    batch_size = images.shape[0]

    # Enhance randomness parameters
    scales = 0.5 + 0.5 * torch.rand(batch_size, device=device)  # [0.5, 1.0]
    rotations = 90 * torch.randn(batch_size, device=device)      # [-90, 90] degrees
    brightness_factors = 0.4 + 0.6 * torch.rand(batch_size, device=device)  # [0.4, 1.0]

    # Generate random positions
    max_offset = 224 - int(patch_size * 1.0)  # Ensure that after scaling, the patch does not go out of bounds
    pos_x = torch.randint(0, max_offset, (batch_size, 1, 1), device=device)
    pos_y = torch.randint(0, max_offset, (batch_size, 1, 1), device=device)

    patched_images = images.clone()
    for i in range(batch_size):
        # Dynamically generate transformation parameters
        current_scale = scales[i]
        current_rotation = rotations[i]
        current_brightness = brightness_factors[i]

        # Create a separate transformation pipeline
        patch_transform = transforms.Compose([
            transforms.Resize(int(patch_size * current_scale)),
            transforms.RandomRotation([current_rotation.item(), current_rotation.item()]),
            transforms.ColorJitter(brightness=current_brightness.item()),
            transforms.RandomPerspective(distortion_scale=0.2, p=0.5)
        ])

        # Apply transformation
        transformed_patch = patch_transform(patch.unsqueeze(0)).squeeze(0)

        # Calculate position
        _, h, w = transformed_patch.shape
        x_start = pos_x[i]
        y_start = pos_y[i]

        # Apply the patch
        patched_images[i, :, y_start:y_start+h, x_start:x_start+w] = transformed_patch

    return torch.clamp(patched_images, 0, 1)


# Define the patch training function

In [23]:

# Improved training process
def train_patch():
    optimizer = optim.Adam([patch], lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=20)

    best_loss = float('inf')
    for epoch in range(epochs):
        epoch_loss = 0.0
        for images in dataloader:
            images = images.to(device)

            # Apply data augmentation
            augmented_images = transforms.Compose([
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.ColorJitter(0.1, 0.1, 0.1),
                transforms.RandomGrayscale(p=0.1)
            ])(images)

            # Apply patch
            patched_images = apply_patch(augmented_images)

            # Get model predictions
            outputs = model(patched_images)

            # Compute loss (including TV regularization)
            ce_loss = -nn.CrossEntropyLoss()(outputs, torch.full((images.size(0),), snake_class, device=device))
            tv_loss = 0.01 * (torch.sum(torch.abs(patch[:, :, :-1] - patch[:, :, 1:])) +
                             torch.sum(torch.abs(patch[:, :-1, :] - patch[:, 1:, :])))
            total_loss = ce_loss + tv_loss

            # Backpropagation
            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()

            # Value clipping
            with torch.no_grad():
                patch.data = torch.clamp(patch, 0, 1)

            epoch_loss += total_loss.item()

        # Adjust learning rate
        avg_loss = epoch_loss / len(dataloader)
        scheduler.step(avg_loss)

        # Save best patch
        if avg_loss < best_loss:
            best_loss = avg_loss
            best_patch = patch.detach().clone()

        if epoch % 20 == 0:
            print(f"Epoch {epoch}, Loss: {avg_loss:.4f}, LR: {optimizer.param_groups[0]['lr']:.6f}")

    return best_patch

# Generate and store the patch

In [32]:
# Genarate the patch
adv_patch = train_patch()

Epoch 0, Loss: 190.3191, LR: 0.010000
Epoch 20, Loss: 79.8714, LR: 0.010000
Epoch 40, Loss: 19.3366, LR: 0.010000
Epoch 60, Loss: -2.4847, LR: 0.010000
Epoch 80, Loss: -7.1960, LR: 0.010000
Epoch 100, Loss: -5.9846, LR: 0.010000
Epoch 120, Loss: -7.3011, LR: 0.010000
Epoch 140, Loss: -9.7988, LR: 0.010000
Epoch 160, Loss: -6.8040, LR: 0.001000
Epoch 180, Loss: -7.5927, LR: 0.000100
Epoch 200, Loss: -11.5499, LR: 0.000100
Epoch 220, Loss: -9.5578, LR: 0.000100
Epoch 240, Loss: -7.3297, LR: 0.000010
Epoch 260, Loss: -8.9205, LR: 0.000001
Epoch 280, Loss: -9.7176, LR: 0.000000
Epoch 300, Loss: -8.8124, LR: 0.000000
Epoch 320, Loss: -7.4622, LR: 0.000000
Epoch 340, Loss: -8.2645, LR: 0.000000
Epoch 360, Loss: -7.8882, LR: 0.000000
Epoch 380, Loss: -12.1546, LR: 0.000000
Epoch 400, Loss: -11.4814, LR: 0.000000
Epoch 420, Loss: -9.1247, LR: 0.000000
Epoch 440, Loss: -10.9024, LR: 0.000000
Epoch 460, Loss: -9.2871, LR: 0.000000
Epoch 480, Loss: -7.9911, LR: 0.000000


In [33]:
final_patch = transforms.ToPILImage()(adv_patch.cpu())

In [34]:
final_patch.save("snake_adversarial_patch.png")

# Test the pacth

In [58]:
# 1.Set up the model
model.eval()  # Set to evaluation mode

# 2. Define preprocessing pipeline (same as used in ImageNet training)
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 3. Load the original image and patch
def load_and_preprocess(image_path):
    img = Image.open(image_path).convert("RGB")
    return preprocess(img).unsqueeze(0)  # Add batch dimension

# Input file paths (modify according to your actual paths)
original_path = "/content/data/TinyImageNet/n01735189/n01735189_1.JPEG"
patch_path = "/content/snake_adversarial_patch.png"

# Load and preprocess the original image
input_tensor = load_and_preprocess(original_path)

# Load and preprocess the patch image
patch_img = Image.open(patch_path).convert("RGB")
patch_tensor = preprocess(patch_img)  # Apply the same preprocessing

# 4. Apply patch at a specified location
adversarial_tensor = input_tensor.clone()

# Define patch position (example: 100x100 region in the bottom right corner)
height, width = 100, 100
adversarial_tensor[:, :, -height:, -width:] = patch_tensor[:, :height, :width]

# 5. Perform predictions
with torch.no_grad():
    # Prediction for the original image
    clean_output = model(input_tensor)
    clean_pred = torch.argmax(clean_output, 1)

    # Prediction for the adversarial sample
    adv_output = model(adversarial_tensor)
    adv_pred = torch.argmax(adv_output, 1)

# Optional: Get class label names
from torchvision.io import read_image
from torchvision.models import ResNet34_Weights
weights = ResNet34_Weights.IMAGENET1K_V1
labels = weights.meta["categories"]
print(f"\nDetailed results:")
print(f"Clean prediction: {labels[clean_pred.item()]} (prob: {F.softmax(clean_output, 1)[0][clean_pred].item():.2%})")
print(f"Adversarial prediction: {labels[adv_pred.item()]} (prob: {F.softmax(adv_output, 1)[0][adv_pred].item():.2%})")


Detailed results:
Clean prediction: garter snake (prob: 74.51%)
Adversarial prediction: garter snake (prob: 18.13%)
